# **Data Preprocessing: Diabetes Health Indicators (BRFSS 2015)**

Notebook ini menjalankan seluruh tahap preprocessing sebelum data masuk ke model MLP. Setiap keputusan diambil berdasarkan temuan di `01_eda.ipynb`, lalu dikunci di `src/preprocessing.py` supaya notebook model, evaluasi, dan aplikasi memakai pipeline yang sama persis.

**Penanggung jawab:** Felix Malvin (231111970)

## **1. Import Library dan Konfigurasi**

In [1]:
import sys
import hashlib
import platform
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy.stats import skew

from src import config
from src import preprocessing as prep

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(config.FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print("Python       :", platform.python_version())
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("Seed         :", config.SEED)

Python       : 3.12.10
pandas       : 3.1.0
scikit-learn : 1.9.1
Seed         : 42


## **2. Memuat Data dan Memeriksa Nilai yang Rusak**

Data dibaca lewat `load_raw_data()`, yang sekaligus mengubah semua kolom dari `float64` ke integer. Setelah itu setiap kolom dicek apakah nilainya masih berada di dalam kode yang sah menurut codebook BRFSS. Nilai di luar kode, misalnya `Age = 14` atau `HighBP = 2`, dianggap sebagai data rusak. Pemeriksaan missing value juga dilakukan di sini.

In [2]:
df_raw = prep.load_raw_data()
print(f"Ukuran data mentah : {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom")
print(f"Total missing value: {int(df_raw.isna().sum().sum())}")

valid_range = {col: (0, 1) for col in config.BINARY_COLS + [config.TARGET_COL]}
valid_range.update({"GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6), "Income": (1, 8),
                    "MentHlth": (0, 30), "PhysHlth": (0, 30)})

cek = []
for col, (lo, hi) in valid_range.items():
    di_luar = ((df_raw[col] < lo) | (df_raw[col] > hi)).sum()
    cek.append({"kolom": col, "rentang_sah": f"{lo}-{hi}", "min": df_raw[col].min(),
                "max": df_raw[col].max(), "nilai_di_luar_rentang": int(di_luar)})
cek = pd.DataFrame(cek)
print(f"Total nilai di luar kode yang sah: {cek['nilai_di_luar_rentang'].sum()}")
cek

Ukuran data mentah : 253,680 baris x 22 kolom
Total missing value: 0
Total nilai di luar kode yang sah: 0


,kolom,rentang_sah,min,max,nilai_di_luar_rentang
0,HighBP,0-1,0,1,0
1,HighChol,0-1,0,1,0
2,CholCheck,0-1,0,1,0
3,Smoker,0-1,0,1,0
4,Stroke,0-1,0,1,0
5,HeartDiseaseorAttack,0-1,0,1,0
6,PhysActivity,0-1,0,1,0
7,Fruits,0-1,0,1,0
8,Veggies,0-1,0,1,0
9,HvyAlcoholConsump,0-1,0,1,0


**Interpretasi:** Tidak ada missing value dan tidak ada satu pun nilai yang keluar dari kode sah codebook BRFSS. Semua fitur biner hanya berisi 0 dan 1, kategori umur tetap di 1 sampai 13, dan jumlah hari sakit tetap di 0 sampai 30. Jadi dataset ini sudah melalui pembersihan dasar dari pembuatnya di Kaggle, sehingga tahap imputasi tidak diperlukan. Satu-satunya kolom yang tidak bisa dicek dengan cara ini adalah `BMI`, karena BMI tidak punya kode tetap dan harus dinilai dari kewajaran nilainya (Bagian 4).